##Attention mechanism

In [1]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Embedding,
    Dense,
    Attention,
    Concatenate
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# =====================================================
# DATASET
# =====================================================

english_sentences = [
    "i love ai",
    "i love deep learning",
    "how are you",
    "good morning"
]

french_sentences = [
    "start j aime ai end",
    "start j aime apprentissage profond end",
    "start comment allez vous end",
    "start bonjour end"
]

# =====================================================
# TOKENIZATION
# =====================================================

eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)

fra_tokenizer = Tokenizer()
fra_tokenizer.fit_on_texts(french_sentences)

encoder_input_sequences = eng_tokenizer.texts_to_sequences(
    english_sentences
)

decoder_sequences = fra_tokenizer.texts_to_sequences(
    french_sentences
)

# =====================================================
# PADDING
# =====================================================

max_encoder_len = max(
    len(seq) for seq in encoder_input_sequences
)

max_decoder_len = max(
    len(seq) for seq in decoder_sequences
)

encoder_input_data = pad_sequences(
    encoder_input_sequences,
    maxlen=max_encoder_len,
    padding="post"
)

decoder_input_data = pad_sequences(
    decoder_sequences,
    maxlen=max_decoder_len,
    padding="post"
)

# =====================================================
# SHIFTED TARGET
# =====================================================

decoder_target_data = np.zeros_like(
    decoder_input_data
)

decoder_target_data[:, :-1] = (
    decoder_input_data[:, 1:]
)

decoder_target_data = np.expand_dims(
    decoder_target_data,
    -1
)

# =====================================================
# VOCAB SIZES
# =====================================================

num_encoder_tokens = (
    len(eng_tokenizer.word_index) + 1
)

num_decoder_tokens = (
    len(fra_tokenizer.word_index) + 1
)

embedding_dim = 64
latent_dim = 64

# =====================================================
# ENCODER
# =====================================================

encoder_inputs = Input(shape=(None,))

encoder_embedding_layer = Embedding(
    num_encoder_tokens,
    embedding_dim
)

encoder_embedding = encoder_embedding_layer(
    encoder_inputs
)

encoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

encoder_outputs, state_h, state_c = (
    encoder_lstm(encoder_embedding)
)

encoder_states = [state_h, state_c]

# =====================================================
# DECODER
# =====================================================

decoder_inputs = Input(shape=(None,))

decoder_embedding_layer = Embedding(
    num_decoder_tokens,
    embedding_dim
)

decoder_embedding = decoder_embedding_layer(
    decoder_inputs
)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

# =====================================================
# ATTENTION
# =====================================================

attention_layer = Attention()

attention_output = attention_layer(
    [decoder_outputs, encoder_outputs]
)

decoder_concat = Concatenate(axis=-1)(
    [decoder_outputs, attention_output]
)

# =====================================================
# OUTPUT LAYER
# =====================================================

decoder_dense = Dense(
    num_decoder_tokens,
    activation="softmax"
)

decoder_outputs = decoder_dense(
    decoder_concat
)

# =====================================================
# TRAINING MODEL
# =====================================================

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# =====================================================
# TRAIN
# =====================================================

model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    epochs=300,
    batch_size=2,
    verbose=1
)

# =====================================================
# ENCODER INFERENCE MODEL
# =====================================================

encoder_model = Model(
    encoder_inputs,
    [encoder_outputs, state_h, state_c]
)

# =====================================================
# DECODER INFERENCE MODEL
# =====================================================

decoder_state_input_h = Input(
    shape=(latent_dim,)
)

decoder_state_input_c = Input(
    shape=(latent_dim,)
)

encoder_outputs_input = Input(
    shape=(max_encoder_len, latent_dim)
)

decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

dec_emb2 = decoder_embedding_layer(
    decoder_inputs
)

decoder_outputs2, state_h2, state_c2 = (
    decoder_lstm(
        dec_emb2,
        initial_state=decoder_states_inputs
    )
)

attention_output2 = attention_layer(
    [decoder_outputs2, encoder_outputs_input]
)

decoder_concat2 = Concatenate(axis=-1)(
    [decoder_outputs2, attention_output2]
)

decoder_outputs2 = decoder_dense(
    decoder_concat2
)

decoder_model = Model(
    [
        decoder_inputs,
        encoder_outputs_input,
        decoder_state_input_h,
        decoder_state_input_c
    ],
    [
        decoder_outputs2,
        state_h2,
        state_c2
    ]
)

# =====================================================
# REVERSE LOOKUP
# =====================================================

reverse_fra_index = {
    v:k
    for k,v in fra_tokenizer.word_index.items()
}

# =====================================================
# DECODE FUNCTION
# =====================================================

def decode_sequence(input_seq):

    encoder_outs, h, c = (
        encoder_model.predict(
            input_seq,
            verbose=0
        )
    )

    start_token = (
        fra_tokenizer.word_index["start"]
    )

    target_seq = np.array(
        [[start_token]]
    )

    decoded_sentence = ""

    while True:

        output_tokens, h, c = (
            decoder_model.predict(
                [
                    target_seq,
                    encoder_outs,
                    h,
                    c
                ],
                verbose=0
            )
        )

        sampled_token_index = np.argmax(
            output_tokens[0, -1, :]
        )

        sampled_word = reverse_fra_index.get(
            sampled_token_index,
            ""
        )

        if sampled_word == "end":
            break

        decoded_sentence += (
            sampled_word + " "
        )

        target_seq = np.array(
            [[sampled_token_index]]
        )

        if len(
            decoded_sentence.split()
        ) > 20:
            break

    return decoded_sentence

# =====================================================
# TEST
# =====================================================

test_sentence = "i love ai"

seq = eng_tokenizer.texts_to_sequences(
    [test_sentence]
)

seq = pad_sequences(
    seq,
    maxlen=max_encoder_len,
    padding="post"
)

prediction = decode_sequence(seq)

print("\nInput :", test_sentence)
print("Output:", prediction)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 64)  │        704 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 64)  │        768 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, None,     │     33,024 │ embedding[0][0]   │
│                     │ 64), (None, 64),  │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │     33,024 │ embedding_1[0][0… │
│                     │ 64), (None, 64),  │            │ lstm[0][1],       │
│                     │ (None, 64)]       │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, None, 64)  │          0 │ lstm_1[0][0],     │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, None, 128) │          0 │ lstm_1[0][0],     │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 12)  │      1,548 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 69,068 (269.80 KB)

 Trainable params: 69,068 (269.80 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.0000e+00 - loss: 2.4864
Epoch 2/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.3750 - loss: 2.4701 
Epoch 3/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3750 - loss: 2.4524
Epoch 4/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3750 - loss: 2.4381
Epoch 5/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.3750 - loss: 2.4163
Epoch 6/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3750 - loss: 2.3943
Epoch 7/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3750 - loss: 2.3680
Epoch 8/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3750 - loss: 2.3353
Epoch 9/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3750 - loss: 2.2853
Epoch 10/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3750 - loss: 2.2336
Epoch 11/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.3750 - loss: 2.1573
Epoch 12/300
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3750